# Notebook 16 — New Dataset: ENSO (El Niño-Southern Oscillation Index)

**Domain:** Atmospheric/oceanographic climate variability
**Source:** NOAA Climate Prediction Center — Oceanic Niño Index (ONI)

## Pre-run prediction

ENSO oscillates between El Niño (warm, positive) and La Niña (cool, negative) phases
on an irregular 3–7 year cycle. It is not a regular sinusoid like keeling_seasonal —
events vary in strength and duration.

**Expected features:**
- `lag1_autocorr` moderate-high (events persist for months)
- `zero_crossings` moderate (crosses zero every ~2–4 years)
- `slope` ≈ 0, `baseline_delta` ≈ 0 (oscillates around zero mean)
- `skewness` slightly positive (El Niño events tend to be stronger than La Niña)
- `spectral_entropy` higher than sunspot (irregular periodicity) but lower than temperature
- `dominant_freq` low (3–7 year cycles in monthly data ≈ 0.012–0.028 cycles/sample)

**Predicted landing:** uncertain. Most likely between sunspot and lynx_hare.
Sunspot: smooth quasi-periodic, low entropy. Lynx_hare: oscillating ecology, moderate entropy.
ENSO is quasi-periodic like sunspot but more irregular, higher entropy.
If HDBSCAN resolves the difference: new class — 'irregular quasi-periodic oscillation'.
If not: collapses into sunspot or lynx_hare cluster.

**This is the most uncertain prediction in Phase 1b.**


In [2]:
import requests
import urllib.request
import json
import shutil
import zipfile
import pandas as pd
import numpy as np
from scipy import stats
from scipy.signal import find_peaks
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler
from itertools import combinations
from sklearn.metrics import adjusted_rand_score
import hdbscan
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TIMEDOM_COLS = ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings', 'slope', 'baseline_delta']
SPECTRAL_COLS = ['dominant_freq', 'spectral_entropy', 'power_low', 'power_mid', 'power_high']
COMBINED_COLS = TIMEDOM_COLS + SPECTRAL_COLS

print('Imports OK')
print(f'Combined feature set ({len(COMBINED_COLS)} features): {COMBINED_COLS}')

Imports OK
Combined feature set (11 features): ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings', 'slope', 'baseline_delta', 'dominant_freq', 'spectral_entropy', 'power_low', 'power_mid', 'power_high']


In [3]:
# ============================================================
# HELPERS
# ============================================================

def zscore_normalize(s):
    s = np.asarray(s, dtype=float)
    std = s.std()
    return (s - s.mean()) / std if std > 0 else s - s.mean()


def baseline_delta(series, frac=0.10):
    n = len(series)
    k = max(1, int(n * frac))
    return float(np.mean(series[-k:]) - np.mean(series[:k]))


def spectral_features_fixed(series):
    """Spectral features computed on original series — NO interpolation.
    dominant_freq is cycles/sample (0–0.5), correctly comparable across series lengths.
    A 12-month annual cycle gives dominant_freq=0.083 whether series is 12 or 480 points.
    """
    s = zscore_normalize(np.asarray(series, dtype=float))
    n = len(s)

    fft_vals = np.fft.rfft(s)
    power    = np.abs(fft_vals) ** 2
    freqs    = np.fft.rfftfreq(n)

    # AC power (exclude DC at index 0)
    power_ac = power[1:]
    freqs_ac = freqs[1:]
    total_ac = power_ac.sum() if power_ac.sum() > 0 else 1.0

    # Dominant frequency (including DC)
    dom_idx  = np.argmax(power)
    dom_freq = float(freqs[dom_idx])

    # Spectral entropy (AC, normalized)
    p_norm = power_ac / total_ac
    p_norm = p_norm[p_norm > 0]
    sp_ent = float(-np.sum(p_norm * np.log(p_norm)))
    sp_ent /= np.log(len(power_ac)) if len(power_ac) > 1 else 1.0

    # Power bands on AC power (low=bottom 20%, mid=20–60%, high=top 40%)
    n_ac   = len(freqs_ac)
    low_end = int(n_ac * 0.20)
    mid_end = int(n_ac * 0.60)
    p_low  = float(power_ac[:low_end].sum()  / total_ac)
    p_mid  = float(power_ac[low_end:mid_end].sum() / total_ac)
    p_high = float(power_ac[mid_end:].sum()  / total_ac)

    return {
        'dominant_freq':   dom_freq,
        'spectral_entropy': sp_ent,
        'power_low':       p_low,
        'power_mid':       p_mid,
        'power_high':      p_high,
    }


def extract_all_features(series):
    """Full 11-feature extraction from a single series."""
    arr = zscore_normalize(np.asarray(series, dtype=float))
    n   = len(arr)
    t   = np.arange(n)
    lag1 = float(np.corrcoef(arr[:-1], arr[1:])[0, 1]) if n > 2 else 0.0
    zc   = float(np.sum(np.diff(np.sign(arr)) != 0) / n)
    slope = float(stats.linregress(t, arr).slope)
    td = {
        'skewness':       float(stats.skew(arr)),
        'kurtosis':       float(stats.kurtosis(arr)),
        'lag1_autocorr':  lag1,
        'zero_crossings': zc,
        'slope':          slope,
        'baseline_delta': baseline_delta(arr),
    }
    sp = spectral_features_fixed(arr)
    return {**td, **sp}


# Sanity check
test = np.sin(np.linspace(0, 4*2*np.pi, 480))  # 4 full cycles in 480 points
f = extract_all_features(test)
print(f'Sine (4 cycles/480pt): dom_freq={f["dominant_freq"]:.4f}  expect={4/480:.4f}')
test2 = np.sin(np.linspace(0, 2*np.pi, 12, endpoint=False))  # 1 cycle in 12 pts
f2 = extract_all_features(test2)
print(f'Sine (1 cycle/12pt):   dom_freq={f2["dominant_freq"]:.4f}  expect={1/12:.4f}')
print('Helpers OK')

Sine (4 cycles/480pt): dom_freq=0.0083  expect=0.0083
Sine (1 cycle/12pt):   dom_freq=0.0833  expect=0.0833
Helpers OK


In [4]:
dest = RAW_DIR / 'owid_covid.csv'
if not dest.exists():
    print('Downloading OWID COVID...')
    r = requests.get('https://github.com/owid/covid-19-data/raw/master/public/data/owid-covid-data.csv', stream=True)
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(8192): f.write(chunk)
df_raw = pd.read_csv(dest, usecols=['location','date','new_cases_smoothed_per_million','continent'], parse_dates=['date'])
df_covid = df_raw.dropna(subset=['continent']).rename(columns={'new_cases_smoothed_per_million':'cases_pm'})

def extract_first_wave(series, max_days=180, min_days=30):
    s = series.fillna(0).values
    starts = np.where(s > 0.5)[0]
    if not len(starts): return None
    start = starts[0]
    wave = s[start:min(start+max_days, len(s))]
    if len(wave) < min_days: return None
    peaks, _ = find_peaks(wave, prominence=wave.max()*0.2)
    if not len(peaks): return None
    wave = wave[:min(peaks[0]+60, len(wave))]
    return wave if len(wave) >= min_days else None

def extract_second_wave(series, min_days=30):
    s = series.fillna(0).values
    peaks, _ = find_peaks(s, prominence=s.max()*0.15, distance=45)
    if len(peaks) < 2: return None
    between = s[peaks[0]:peaks[1]]
    start = peaks[0] + np.argmin(between)
    wave = s[start:min(peaks[1]+60, len(s))]
    return wave if len(wave) >= min_days else None

records = []
for country, grp in df_covid.groupby('location'):
    grp = grp.sort_values('date')
    for fn, ds in [(extract_first_wave,'covid_first_wave'), (extract_second_wave,'covid_second_wave')]:
        w = fn(grp['cases_pm'])
        if w is not None:
            feats = extract_all_features(w)
            feats.update({'country': country, 'dataset': ds, 'n_points': len(w)})
            records.append(feats)
df_covid_all = pd.DataFrame(records)
print(df_covid_all['dataset'].value_counts().to_dict())

{'covid_second_wave': 209, 'covid_first_wave': 202}


In [5]:
dest = RAW_DIR / 'sunspot_monthly.csv'
if not dest.exists():
    dest.write_bytes(requests.get('https://www.sidc.be/silso/DATA/SN_m_tot_V2.0.csv').content)
df_ss = pd.read_csv(dest, sep=';', header=None,
                    names=['year','month','frac_year','monthly_mean','monthly_sd','n_obs','definitive'],
                    na_values=[-1])
df_ss = df_ss.dropna(subset=['monthly_mean'])
df_ss['smooth'] = df_ss['monthly_mean'].rolling(13, center=True).mean()
series_full = df_ss['smooth'].bfill().ffill().values
smoothed = pd.Series(series_full).rolling(25, center=True).mean().bfill().ffill().values
minima, _ = find_peaks(-smoothed, distance=80)
cycles = {}
for i in range(len(minima)-1):
    c = series_full[minima[i]:minima[i+1]]
    if len(c) >= 80:
        cycles[f'cycle_{i+1}_{int(df_ss["year"].iloc[minima[i]])}'] = c
records = []
for name, c in cycles.items():
    feats = extract_all_features(c)
    feats.update({'country': name, 'dataset': 'sunspot_cycle', 'n_points': len(c)})
    records.append(feats)
df_ss_all = pd.DataFrame(records)
print(f'Sunspot: {len(df_ss_all)}  |  mean dom_freq={df_ss_all["dominant_freq"].mean():.4f}')

Sunspot: 24  |  mean dom_freq=0.0077


In [6]:
df_lh = pd.read_csv(Path('../datasets/lynx_hare/lynx_hare.csv'))
year_col = [c for c in df_lh.columns if c.lower()=='year'][0]
species_cols = [c for c in df_lh.columns if c.lower()!='year']
window_size = 10
series_dict = {}
for sp in species_cols:
    full = df_lh[sp].values.astype(float)
    series_dict[f'{sp}_full'] = full
    for start in range(len(full)-window_size+1):
        series_dict[f'{sp}_w{start}_{df_lh[year_col].iloc[start]}'] = full[start:start+window_size]
records = []
for name, s in series_dict.items():
    feats = extract_all_features(s)
    feats.update({'country': name, 'dataset': 'lynx_hare', 'n_points': len(s)})
    records.append(feats)
df_lh_all = pd.DataFrame(records)
print(f'Lynx-hare: {len(df_lh_all)}')

Lynx-hare: 26


In [7]:
dest = RAW_DIR / 'keeling_monthly.csv'
if not dest.exists():
    dest.write_bytes(requests.get('https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv').content)
co2 = pd.read_csv(dest, comment='#', header=None,
                  names=['year','month','decimal_date','average','deseasonalized','ndays','sdev','unc'])
for col in ['year','month','average']:
    co2[col] = pd.to_numeric(co2[col], errors='coerce')
co2 = co2.dropna(subset=['year','month','average'])
co2 = co2[co2['average']>0].copy()
co2.index = pd.to_datetime({'year':co2['year'].astype(int),'month':co2['month'].astype(int),'day':1})
result = seasonal_decompose(co2['average'], model='additive', period=12, extrapolate_trend='freq')
seasonal_vals = result.seasonal.dropna().values
trend_vals    = result.trend.dropna().values
start_year    = co2.index.min().year
series_dict = {}
for i in range(len(seasonal_vals)//12):
    seg = seasonal_vals[i*12:(i+1)*12]
    if len(seg)==12: series_dict[f'keeling_seasonal_{start_year+i}'] = seg
for i in range(0, len(trend_vals)-120, 12):
    series_dict[f'keeling_trend_{start_year+i//12}'] = trend_vals[i:i+120]
records = []
for name, s in series_dict.items():
    feats = extract_all_features(s)
    feats.update({'country': name,
                  'dataset': 'keeling_seasonal' if 'seasonal' in name else 'keeling_trend',
                  'n_points': len(s)})
    records.append(feats)
df_k_all = pd.DataFrame(records)
print(df_k_all['dataset'].value_counts().to_dict())
for ds in ['keeling_seasonal','keeling_trend']:
    sub = df_k_all[df_k_all['dataset']==ds]
    print(f'  {ds}: dom_freq={sub["dominant_freq"].mean():.4f}  entropy={sub["spectral_entropy"].mean():.4f}  bd={sub["baseline_delta"].mean():.3f}')

{'keeling_seasonal': 68, 'keeling_trend': 58}
  keeling_seasonal: dom_freq=0.0833  entropy=0.1535  bd=-0.337
  keeling_trend: dom_freq=0.0083  entropy=0.3885  bd=3.111


In [8]:
dest = RAW_DIR / 'temperature_anomaly.csv'
if not dest.exists():
    print('Downloading Berkeley Earth temperature...')
    r = requests.get('https://berkeley-earth-temperature.s3.amazonaws.com/Global/Land_and_Ocean_summary.txt',
                     headers={'User-Agent':'Mozilla/5.0'}, timeout=30)
    r.raise_for_status()
    dest.write_bytes(r.content)
with open(dest) as f: raw = f.read()
if raw.lstrip().startswith('%'):
    rows = []
    for line in raw.splitlines():
        if line.strip() and not line.strip().startswith('%'):
            parts = line.split()
            if len(parts)>=2:
                try: rows.append({'year':int(float(parts[0])),'anomaly':float(parts[1])})
                except ValueError: pass
    df_temp = pd.DataFrame(rows).dropna()
else:
    lines = raw.splitlines()
    hidx = next(i for i,l in enumerate(lines) if 'Year' in l and 'J-D' in l)
    df_temp = pd.read_csv(dest, skiprows=hidx, na_values=['***','****'])
    df_temp = df_temp[['Year','J-D']].rename(columns={'Year':'year','J-D':'anomaly'})
    df_temp['year'] = pd.to_numeric(df_temp['year'],errors='coerce')
    df_temp['anomaly'] = pd.to_numeric(df_temp['anomaly'],errors='coerce')
    df_temp = df_temp.dropna()
    df_temp['year'] = df_temp['year'].astype(int)
values = df_temp['anomaly'].values
years  = df_temp['year'].values
window, step = 20, 5
records = []
for i in range(0, len(values)-window, step):
    s = values[i:i+window]
    feats = extract_all_features(s)
    feats.update({'country':f'temp_{years[i]}','dataset':'temperature','n_points':len(s)})
    records.append(feats)
df_temp_all = pd.DataFrame(records)
print(f'Temperature: {len(df_temp_all)}  mean bd={df_temp_all["baseline_delta"].mean():.3f}')

Temperature: 31  mean bd=0.997


In [9]:
dest_zip = RAW_DIR / 'ECGFiveDays.zip'
dest_dir = RAW_DIR / 'ECGFiveDays'
if dest_dir.exists() and list(dest_dir.rglob('*.ts')):
    print(f'Cached: {dest_dir}')
elif dest_zip.exists():
    if dest_dir.exists(): shutil.rmtree(dest_dir)
    with zipfile.ZipFile(dest_zip) as z: z.extractall(dest_dir)
    print(f'Extracted from zip')
else:
    raise RuntimeError('ECGFiveDays.zip not found — place in data/raw/')
ts_files = list(dest_dir.rglob('*.ts'))
print(f'.ts files: {[f.name for f in ts_files]}')

def parse_ts_file(path):
    series_list, labels = [], []
    in_data = False
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.lower()=='@data': in_data=True; continue
            if in_data and not line.startswith('@'):
                if ':' in line:
                    data_part, label = line.rsplit(':',1)
                    values = [float(x) for x in data_part.split(',') if x.strip()]
                else:
                    parts = line.split()
                    try: values=[float(x) for x in parts[:-1]]; label=parts[-1]
                    except (ValueError,IndexError): continue
                if values: series_list.append(np.array(values)); labels.append(label.strip())
    return series_list, labels

all_series, all_labels = [], []
for f in ts_files:
    s, l = parse_ts_file(f)
    all_series.extend(s); all_labels.extend(l)
print(f'ECG segments: {len(all_series)}')
records = []
for i, (s, label) in enumerate(zip(all_series, all_labels)):
    feats = extract_all_features(s)
    feats.update({'country':f'ecg_{i}_c{label}','dataset':'ecg','n_points':len(s)})
    records.append(feats)
df_ecg_all = pd.DataFrame(records)
print(f'ECG dom_freq={df_ecg_all["dominant_freq"].mean():.4f}  kurtosis={df_ecg_all["kurtosis"].mean():.3f}')

Cached: ../data/raw/ECGFiveDays
.ts files: ['ECGFiveDays_TRAIN.ts', 'ECGFiveDays_TEST.ts']
ECG segments: 884
ECG dom_freq=0.0175  kurtosis=15.165


In [ ]:
STATIONS = {
    '01350000':'mohawk_ny',    '01427207':'delaware_ny', '01491000':'choptank_md',
    '02087500':'neuse_nc',     '02339500':'flint_ga',    '03049000':'allegheny_pa',
    '03611500':'ohio_il',      '05054000':'red_nd',      '05378500':'mississippi_mn',
    '05420500':'mississippi_ia','06289000':'bighorn_mt', '06600000':'missouri_ia',
    '07022000':'mississippi_mo','07289000':'mississippi_ms','08220000':'riogrande_co',
    '09180000':'colorado_ut',  '09380000':'colorado_az', '11530000':'klamath_ca',
    '12040000':'queets_wa',    '12374250':'clearwater_id','14105700':'columbia_or',
    '14179000':'willamette_or','06354000':'cannonball_nd','02479155':'escatawpa_ms',
    '01096500':'nashua_ma',
}

def fetch_monthly_flow(site_id, start='1980-01-01', end='2020-12-31'):
    url = (f'https://waterservices.usgs.gov/nwis/dv/?format=json'
           f'&sites={site_id}&parameterCd=00060&statCd=00003&startDT={start}&endDT={end}')
    try:
        req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=45) as resp:
            data = json.loads(resp.read())
        series = data['value']['timeSeries']
        if not series: return None
        recs = []
        for v in series[0]['values'][0]['value']:
            try: recs.append({'date':pd.Timestamp(v['dateTime'][:10]),'flow':float(v['value'])})
            except (ValueError,KeyError): pass
        if len(recs)<365: return None
        df = pd.DataFrame(recs).set_index('date')
        monthly = df['flow'].resample('MS').mean().dropna()
        return monthly if len(monthly)>=60 else None
    except Exception: return None

print(f'Fetching {len(STATIONS)} USGS stations...')
flows = {}
for site_id, name in STATIONS.items():
    s = fetch_monthly_flow(site_id)
    if s is not None: flows[name]=s; print(f'  OK {name}')
    else: print(f'  FAIL {name}')
print(f'Loaded: {len(flows)} stations')

records = []
for name, series in flows.items():
    log_flow = np.log1p(series.values.astype(float))
    feats = extract_all_features(log_flow)
    feats.update({'country':name,'dataset':'streamflow','n_points':len(log_flow)})
    records.append(feats)
df_sf_all = pd.DataFrame(records)
print(f'Streamflow dom_freq={df_sf_all["dominant_freq"].mean():.4f}  (expect ~0.083 — fixed from nb12\'s 0.353)')

Fetching 25 USGS stations...
  OK mohawk_ny
  OK delaware_ny
  OK choptank_md
  OK neuse_nc
  OK flint_ga
  OK allegheny_pa
  OK ohio_il
  OK red_nd
  OK mississippi_mn
  OK mississippi_ia
  OK bighorn_mt
  OK missouri_ia
  OK mississippi_mo
  OK mississippi_ms
  OK riogrande_co
  OK colorado_ut
  OK colorado_az
  OK klamath_ca
  FAIL queets_wa
  OK clearwater_id
  OK columbia_or


In [ ]:
# ============================================================
# NEW DATASET: ENSO Oceanic Nino Index (ONI)
# NOAA CPC — 3-month running mean of ERSST.v5 anomalies
# ============================================================
dest = RAW_DIR / 'oni_enso.txt'
if not dest.exists():
    print('Downloading NOAA ONI...')
    r = requests.get(
        'https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt',
        headers={'User-Agent': 'Mozilla/5.0'}, timeout=30
    )
    r.raise_for_status()
    dest.write_bytes(r.content)

rows = []
with open(dest) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('SEAS') or line.startswith('YR'):
            continue
        parts = line.split()
        if len(parts) >= 3:
            try:
                yr  = int(parts[0])
                val = float(parts[2])   # ANOM column
                rows.append({'year': yr, 'oni': val})
            except (ValueError, IndexError):
                pass

df_oni = pd.DataFrame(rows).dropna()
print(f'ONI records: {len(df_oni)}  ({df_oni["year"].min()}–{df_oni["year"].max()})')
print(df_oni.head(5))


In [ ]:
# ONI is already a 3-month running mean — use the full series as one signal
# Also slide windows (length 36 = 3 years, step 6) to get multiple instances
values = df_oni['oni'].values
print(f'Full series: {len(values)} monthly values')
print(f'Range: {values.min():.2f} to {values.max():.2f}')

window, step = 36, 6   # 3-year windows, 6-month step
records = []
for i in range(0, len(values) - window, step):
    s = values[i:i+window]
    feats = extract_all_features(s)
    yr = df_oni['year'].iloc[i] if i < len(df_oni) else ''
    feats.update({'country': f'oni_{yr}', 'dataset': 'enso_oni', 'n_points': len(s)})
    records.append(feats)

df_new = pd.DataFrame(records)
print(f'ENSO windows: {len(df_new)}')
TIMEDOM_COLS = ['skewness','kurtosis','lag1_autocorr','zero_crossings','slope','baseline_delta']
SPECTRAL_COLS = ['dominant_freq','spectral_entropy','power_low','power_mid','power_high']
print(df_new[TIMEDOM_COLS + SPECTRAL_COLS].mean().round(4).to_string())


In [ ]:
# ============================================================
# CLUSTER: all 9 original + sea_level
# ============================================================
TIMEDOM_COLS = ['skewness','kurtosis','lag1_autocorr','zero_crossings','slope','baseline_delta']
SPECTRAL_COLS = ['dominant_freq','spectral_entropy','power_low','power_mid','power_high']
ALL_COLS = TIMEDOM_COLS + SPECTRAL_COLS

df_all_ext = pd.concat([df_covid_all, df_ss_all, df_lh_all, df_k_all,
                         df_temp_all, df_ecg_all, df_sf_all, df_new], ignore_index=True)
df_clean = df_all_ext.dropna(subset=ALL_COLS).copy()
NEW_DS = df_new['dataset'].iloc[0]

print(f'Total instances: {len(df_clean)}')
print(df_clean['dataset'].value_counts().to_string())
print()
print(f'=== New dataset feature profile: {NEW_DS} ===')
print(df_clean[df_clean['dataset']==NEW_DS][ALL_COLS].mean().round(4).to_string())
print()
print('=== Closest existing datasets by TD centroid distance ===')
X_td = StandardScaler().fit_transform(df_clean[TIMEDOM_COLS].values)
datasets = sorted(df_clean['dataset'].unique())
centroids = {ds: X_td[(df_clean['dataset']==ds).values].mean(axis=0) for ds in datasets}
new_c = centroids[NEW_DS]
dists = {ds: float(np.linalg.norm(centroids[ds]-new_c)) for ds in datasets if ds != NEW_DS}
for ds, d in sorted(dists.items(), key=lambda x: x[1]):
    print(f'  {ds:25s}: {d:.3f}')


In [ ]:
# ============================================================
# SCORE PREDICTION
# ============================================================
import hdbscan
import umap
from sklearn.metrics import adjusted_rand_score

domain_int = pd.factorize(df_clean['dataset'])[0]
X = StandardScaler().fit_transform(df_clean[TIMEDOM_COLS].values)
labels = hdbscan.HDBSCAN(min_cluster_size=8, min_samples=3).fit_predict(X)
n_cl  = len(set(labels)) - (1 if -1 in labels else 0)
noise = (labels==-1).sum()
non_noise = labels != -1
ari = adjusted_rand_score(domain_int[non_noise], labels[non_noise])
print(f'Clusters: {n_cl}  Noise: {noise} ({100*noise/len(labels):.1f}%)  ARI: {ari:.3f}')
print()

# Where did the new dataset land?
df_clean['cluster'] = labels
new_mask = df_clean['dataset'] == NEW_DS
new_clusters = df_clean.loc[new_mask, 'cluster'].value_counts()
majority = new_clusters.index[0]
pct = 100 * new_clusters.iloc[0] / new_mask.sum()
print(f'{NEW_DS} -> Cluster {majority} ({pct:.0f}% of instances)')
print()

# Which existing datasets share that cluster?
cluster_members = df_clean[df_clean['cluster']==majority]['dataset'].value_counts()
print(f'Cluster {majority} members:')
print(cluster_members.to_string())


---
## Findings to record

- **Closest existing dataset by TD centroid:** _(fill)_ — predicted: sunspot or lynx_hare
- **Cluster landing:** _(fill)_ — predicted: between sunspot and lynx_hare, or new class
- **Prediction status:** CONFIRMED / UNEXPECTED
- **Key feature values:** lag1_autocorr=_, zero_crossings=_, spectral_entropy=_, dominant_freq=_
- **Did it form a new class?** _(yes/no)_
- **Surprise (if any):** _(fill)_
